# Week 96, Day 1: Contribution sprint

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w096d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('units.py', 'w', encoding='utf-8') as f:
    f.write('"""Conversions for walkers.\n\n>>> km_to_miles(10)\n6.21\n"""\n\n\ndef km_to_miles(km):\n    """Convert kilometres to miles, rounded to 2 places.\n\n    >>> km_to_miles(5)\n    3.11\n    """\n    return round(km * 0.621371, 2)\n\n\ndef pace(minutes, km):\n    """Minutes per kilometre.\n\n    >>> pace(50, 10)\n    5.0\n    >>> pace(45, 0)\n    0.0\n    """\n    if km == 0:\n        raise ValueError("distance must be positive")\n    return minutes / km\n\n\ndef format_duration(minutes):\n    """Write minutes as hours and minutes.\n\n    >>> format_duration(125)\n    \'2 h 5 min\'\n    """\n    return f"{minutes // 60} h {minutes % 60:02d} min"\n')
with open('units.rst', 'w', encoding='utf-8') as f:
    f.write(':mod:`units`: conversions for walkers\n=====================================\n\n.. function:: km_to_miles(km)\n\n   Convert kilometres to miles, rounded to 2 places.\n\n.. function:: pace(minutes, distance)\n\n   Return minutes per kilometre. See also :func:`speed`.\n\n.. function:: format_duration(minutes, short=False)\n\n   Write *minutes* as hours and minutes, as :func:`format_duration` does.\n\n.. function:: miles_to_km(miles)\n\n   Convert miles to kilometres.\n')

**Predict before you run:** what will this print?



In [ ]:
from units import format_duration, pace

print(repr(format_duration(125)))
try:
    pace(45, 0)
except ValueError as e:
    print("ValueError:", e)

> **Key idea.** Sprint in a steady rhythm: small issue, reproduce, change, test, self-review, PR, next; keep a log. Decide which side is wrong, code or docs, before fixing; ask if unsure. Find drift three ways: doctest for examples, directives against the syntax tree for signatures, roles against defined names for references. Prove a change is docs only by comparing syntax trees with docstrings removed.

### Exercise: Examples that drifted

Write `doc_mismatches(source)`: run the examples in the module's docstring and in every function's and class's docstring (found with `ast`), in a namespace made by executing the source, and return `(NAME, EXAMPLE, DOCUMENTED, ACTUAL)` for each failing example, in order: NAME is `"module"` or the function's name, EXAMPLE the example's source stripped, DOCUMENTED its expected output stripped, and ACTUAL the real output stripped, or `"TYPE: MESSAGE"` if it raised.

In [ ]:
import ast
import doctest

def doc_mismatches(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_units():\n    \"two examples that drifted\"\n    assert doc_mismatches(open(\"units.py\").read()) == [\n        (\"pace\", \"pace(45, 0)\", \"0.0\", \"ValueError: distance must be positive\"),\n        (\"format_duration\", \"format_duration(125)\", \"'2 h 5 min'\", \"'2 h 05 min'\")]\n    assert doc_mismatches('\"\"\"\\n>>> 1 + 1\\n2\\n\"\"\"\\n') == []")

### Exercise: Signatures that drifted

Write `signature_drift(source, rst)`: read each `.. function:: NAME(PARAMS)` line of the reST (parameter names only, dropping any `=default`), and compare with the public top-level functions in the source. Report, for documented functions in order, `"NAME: documented but not in the code"` or `"NAME: documented as (A, B), code has (A, C)"`; then `"NAME: not documented"` for code functions missing from the page.

In [ ]:
import ast
import re

def signature_drift(source, rst):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_units():\n    \"the reference page against the code\"\n    assert signature_drift(open(\"units.py\").read(), open(\"units.rst\").read()) == [\n        \"pace: documented as (minutes, distance), code has (minutes, km)\",\n        \"format_duration: documented as (minutes, short), code has (minutes)\",\n        \"miles_to_km: documented but not in the code\"]\n    assert signature_drift(\"def a(x):\\n    pass\\n\\ndef _b():\\n    pass\\n\", \"\") == [\"a: not documented\"]")

### Exercise: References to nothing

Write `unknown_references(rst, defined)`: return, sorted and without repeats, the targets of `:func:` roles (``:func:`NAME` ``, where NAME may contain dots) that aren't in the set `defined`.

In [ ]:
import re

def unknown_references(rst, defined):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_refs():\n    \"what Sphinx would warn about\"\n    assert unknown_references(open(\"units.rst\").read(), {\"km_to_miles\", \"pace\", \"format_duration\"}) == [\"speed\"]\n    assert unknown_references(\":func:`a.b` and :func:`a.b` and :func:`c`\", {\"c\"}) == [\"a.b\"]")

### Exercise: Provably documentation only

Write `docs_only(old, new)`: return whether the two sources are the same program once every docstring (of the module, functions and classes) is removed, by comparing `ast.dump` of their syntax trees. A body that only had a docstring becomes `pass`.

In [ ]:
import ast

def docs_only(old, new):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_docs_only():\n    \"nothing the program does has changed\"\n    src = open(\"units.py\").read()\n    assert docs_only(src, src.replace(\"'2 h 5 min'\", \"'2 h 05 min'\"))\n    assert docs_only(src, src.replace('\"\"\"Minutes per kilometre.', '\"\"\"Return minutes per kilometre.'))\n    assert not docs_only(src, src.replace(\"0.621371\", \"0.6214\"))\n    assert docs_only(\"def f():\\n    '''Old.'''\\n\", \"def f():\\n    pass\\n\")\n    assert docs_only(src, src.replace(\"    return minutes / km\", \"    return minutes / km  # per km\"))")

### Bug hunt: A default that hid a parameter

The directive's parameters are split on commas, but the default is kept, so `short=False` is compared with `short` and this prints `['minutes', 'short=False']`. Drop the defaults so it prints `['minutes', 'short']`.

In [ ]:
line = ".. function:: format_duration(minutes, short=False)"
params = line[line.index("(") + 1:line.rindex(")")]
print([p.strip() for p in params.split(",")])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"names only\"\n    expect_output(\"['minutes', 'short']\")")

### Bug hunt: Docs only, except for one comment

Comments aren't in the syntax tree, so adding one changes nothing the program does, but comparing the source text says otherwise and this prints `False`. Compare syntax trees so it prints `True`.

In [ ]:
import ast

old = "def pace(minutes, km):\n    return minutes / km\n"
new = "def pace(minutes, km):\n    return minutes / km  # per km\n"
print(old == new)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"ast.dump\"\n    expect_output(\"True\")")

### Puzzle: Removing docstrings

Arrange the function that removes docstrings before comparing trees. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the reST directive that documents a function, with its two colons
q = ___   # the role that links to a function, with its colons
r = ___   # format_duration(125) with the code as it is (a string)
s = ___   # True or False: comments appear in ast.dump
t = ___   # the label a core developer adds for a small docs fix that needs no NEWS entry

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \".. function::\" and q == \":func:\" and r == \"2 h 05 min\", \"p, q and r\"\n    assert s is False and t == \"skip news\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: correcting the examples

Write `fix_examples(source)` with `doc_mismatches` (given). For each mismatch, find the line in `source` whose stripped text is `>>> EXAMPLE`, and replace the expected-output lines after it (up to a blank line, a line starting with `>>>`, or a line containing `"""`, stripped) with the actual output at the same indentation; for an exception, write three lines: `Traceback (most recent call last):`, `...` indented four spaces further, and `TYPE: MESSAGE`. Return `(new_source, notes)`, with a note `"NAME: EXAMPLE now shows ACTUAL"` per fix.

In [ ]:
import ast
import doctest

class Collector(doctest.DocTestRunner):
    def __init__(self):
        super().__init__(verbose=False)
        self.found = []
    def report_failure(self, out, test, example, got):
        self.found.append((test.name, example.source.strip(), example.want.strip(), got.strip()))
    def report_unexpected_exception(self, out, test, example, exc_info):
        self.found.append((test.name, example.source.strip(), example.want.strip(), f"{exc_info[0].__name__}: {exc_info[1]}"))

def doc_mismatches(source):
    tree = ast.parse(source)
    docs = [("module", ast.get_docstring(tree))]
    docs += [(n.name, ast.get_docstring(n)) for n in ast.walk(tree) if isinstance(n, (ast.FunctionDef, ast.ClassDef))]
    globs = {}
    exec(compile(source, "module", "exec"), globs)
    runner = Collector()
    for name, doc in docs:
        if doc:
            runner.run(doctest.DocTestParser().get_doctest(doc, dict(globs), name, None, 0), out=lambda s: None)
    return runner.found

def doc_mismatches(source):
    tree = ast.parse(source)
    docs = [("module", ast.get_docstring(tree))]
    docs += [(n.name, ast.get_docstring(n)) for n in ast.walk(tree) if isinstance(n, (ast.FunctionDef, ast.ClassDef))]
    globs = {}
    exec(compile(source, "module", "exec"), globs)
    runner = Collector()
    for name, doc in docs:
        if doc:
            runner.run(doctest.DocTestParser().get_doctest(doc, dict(globs), name, None, 0), out=lambda s: None)
    return runner.found

def fix_examples(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_units():\n    \"corrected, and provably docs only\"\n    src = open(\"units.py\").read()\n    fixed, notes = fix_examples(src)\n    assert notes == [\"pace: pace(45, 0) now shows ValueError: distance must be positive\",\n                     \"format_duration: format_duration(125) now shows '2 h 05 min'\"]\n    assert doc_mismatches(fixed) == []\n    assert \"    >>> pace(45, 0)\\n    Traceback (most recent call last):\\n        ...\\n    ValueError: distance must be positive\\n\" in fixed\n    assert \"    >>> format_duration(125)\\n    '2 h 05 min'\\n\" in fixed\n    import ast\n    strip = lambda s: [n for n in ast.walk(ast.parse(s)) if isinstance(n, ast.Return)]\n    assert len(strip(fixed)) == len(strip(src)) and fixed.count(\"\\n\") == src.count(\"\\n\") + 2\n\ndef test_nothing():\n    \"already right\"\n    src = '\"\"\"\\n>>> 1 + 1\\n2\\n\"\"\"\\n'\n    assert fix_examples(src) == (src, [])")

## Warm-up

Compare that with the docstrings in `units.py`: one says `'2 h 5 min'`, the other says `pace(45, 0)` is `0.0`. Here the code is right (zero-padded minutes, and an error for a zero distance) and the documentation is wrong. Deciding which side is wrong is the first judgement in every docs fix; when it's unclear, ask on the issue.

## Sprinting

A **sprint** is a short, focused stretch of contributing, often in a group (PyCon and many local meetups run sprints where maintainers help newcomers in person). The rhythm is the one from last week, compressed: pick a small issue, reproduce, change, test, self-review, open the pull request, and move on while you wait for review. Keep a log as you go (issue, branch, pull request, status): you'll need it on Day 5.

## Three kinds of drift

- **Examples** that no longer match: doctest finds them, as on Day 1 of last week.
- **Signatures** that no longer match: the reference page says `pace(minutes, distance)` but the code says `pace(minutes, km)`; a documented parameter that was removed; a function documented but deleted. CPython's reference pages use reStructuredText directives (`.. function:: name(args)`), which a regular expression can read well enough to compare with the syntax tree.
- **References** to things that don't exist: a ``:func:`speed` `` role pointing at nothing. Sphinx warns about these when it builds the docs (`make html` in `Doc/`, with `-n` for "nitpicky" warnings).

## Docs only, provably

A documentation-only pull request is quicker to review if it's **provably** documentation only. Compare the syntax trees of the old and new code with every docstring removed: if they're identical, nothing the program does has changed. Small documentation fixes in CPython often don't need a NEWS entry (a core developer adds `skip news`), and they're backported to the bugfix branches.

## Practice

## Quiz

## Daily challenge

## Your sprint

1. **Run doctest** over a standard library module's docstrings and its documentation page (`./python -m doctest Lib/NAME.py`, and `make doctest` in `Doc/`), or a library you use, and look for examples that drifted.
2. **Compare signatures** between a reference page and the code with your `signature_drift`.
3. **Fix one** (a docs issue labelled `docs` and `easy` is ideal), prove it's docs only, and open the pull request.
4. **Log it**: issue, branch, PR, status.

## Before you go

Tomorrow's sprint is a test-only contribution: finding code no test runs, and writing the tests.